# R1000 Quant Engine - Colab Runner
GitHub에서 최신 코드를 가져와 실행합니다.

**순서**: Cell 1 → Cell 2 → Cell 3 → Cell 4 (순서대로 실행)

In [ ]:
# ===== Cell 1: 의존성 설치 + GitHub에서 최신 코드 가져오기 =====
!pip install -q yfinance fredapi catboost scikit-learn requests pandas numpy

import os, shutil

REPO_DIR = "/content/r1000-quant-engine"
REPO_URL = "https://github.com/wscha231/r1000-quant-engine.git"

if os.path.exists(REPO_DIR):
    # 이미 클론되어 있으면 최신 코드로 업데이트
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/master
    print("[OK] 최신 코드로 업데이트 완료")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("[OK] GitHub에서 클론 완료")

# 최신 커밋 확인
!cd {REPO_DIR} && git log --oneline -3
print("\n[OK] Cell 1 완료 — 다음 Cell 2 실행")

In [ ]:
# ===== Cell 2: Google Drive 마운트 + 이전 데이터 스마트 정리 =====
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/r1000_top30_institutional"

# ============================================================
#   FRESH_START = True  → 호환 안 되는 캐시만 삭제 (가격/companyfacts 보존)
#   FRESH_START = False → 기존 데이터 전부 재활용 (일반 업데이트)
#   NUKE_ALL    = True  → 정말 전부 삭제 (최초 실행 or 완전 초기화)
# ============================================================
FRESH_START = True
NUKE_ALL = False

# 보존할 폴더/파일 (재다운로드 오래 걸리는 것들)
KEEP_ON_FRESH = {
    "cache_prices",        # 가격 데이터 — 구조 변경 없음, 재다운 30분+
    "data_raw",            # 원본 데이터 (유니버스 멤버십 등)
    "companyfacts.zip",    # SEC 벌크 ~1GB, 재다운 10분+
}

# 삭제 대상 (코드 구조 변경으로 호환 안 됨)
DELETE_ON_FRESH = {
    "__pycache__",         # Python 캐시
    "catboost_info",       # CatBoost 학습 로그
    "feature_store",       # 이전 피처 (컬럼 구조 변경)
    "models",              # 이전 모델 (재학습 필요)
    "outputs",             # 이전 결과물
    "baseline",            # 코드 감사 파일
    "cache_fsds",          # SEC FSDS 캐시 (구조 변경)
    "cache_live_fund",     # 라이브 펀더멘탈 캐시
    "cache_macro",         # 매크로 캐시
    "cache_misc",          # 기타 캐시
    "cache_sec_actual",    # SEC actual 캐시
    "reports",             # 이전 리포트
}

if os.path.exists(DATA_DIR):
    if NUKE_ALL:
        old_size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fns in os.walk(DATA_DIR) for f in fns
        ) / (1024 * 1024)
        shutil.rmtree(DATA_DIR)
        print(f"[NUKE] 전체 삭제 완료 ({old_size:.0f} MB)")
    elif FRESH_START:
        deleted, kept = [], []
        for item in os.listdir(DATA_DIR):
            path = os.path.join(DATA_DIR, item)
            if item in KEEP_ON_FRESH:
                if os.path.isdir(path):
                    size = sum(os.path.getsize(os.path.join(dp, f))
                               for dp, _, fns in os.walk(path) for f in fns) / (1024*1024)
                else:
                    size = os.path.getsize(path) / (1024*1024)
                kept.append(f"  [보존] {item} ({size:.0f} MB)")
            elif item in DELETE_ON_FRESH or item.endswith((".py", ".pyc")):
                if os.path.isdir(path):
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                deleted.append(f"  [삭제] {item}")
            else:
                kept.append(f"  [유지] {item} (목록에 없음)")
        print("[FRESH START] 스마트 정리 완료:")
        for line in deleted:
            print(line)
        for line in kept:
            print(line)
    else:
        print("[INFO] 기존 데이터 전부 유지 (재활용 모드)")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"\n[OK] 데이터 경로: {DATA_DIR}")
print("[OK] Cell 2 완료 — 다음 Cell 3 실행")

In [ ]:
# ===== Cell 3: 엔진 설정 =====
import sys, os
from datetime import datetime, timedelta

REPO_DIR = "/content/r1000-quant-engine"

# 1) 레포 존재 확인
if not os.path.exists(REPO_DIR):
    raise RuntimeError(f"[ERROR] {REPO_DIR} 없음 — Cell 1을 먼저 실행하세요!")

# 2) 엔진 파일 존재 확인
engine_file = os.path.join(REPO_DIR, "r1000_top30_institutional.py")
if not os.path.exists(engine_file):
    print(f"[ERROR] {engine_file} 없음!")
    print(f"레포 내 파일 목록:")
    for f in os.listdir(REPO_DIR):
        print(f"  {f}")
    raise RuntimeError("엔진 파일을 찾을 수 없습니다")

# 3) sys.path에 추가
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 4) 모듈 캐시 제거 (코드 업데이트 반영)
for mod_name in list(sys.modules.keys()):
    if "r1000" in mod_name:
        del sys.modules[mod_name]

print(f"[OK] 엔진 파일 확인: {engine_file}")

from r1000_top30_institutional import EngineConfig, run_default_pipeline

# ============================================================
#   설정 — 필요시 수정
# ============================================================
us_yesterday = (datetime.utcnow() - timedelta(hours=5, days=1)).strftime("%Y-%m-%d")

cfg = EngineConfig(
    base_dir="/content/drive/MyDrive/r1000_top30_institutional",
    end_date=us_yesterday,

    # API 키
    fred_api_key="8d92fb5a5de226657d912fe0284dfc00",

    # 데이터 갱신 주기 (일)
    live_refresh_days=7,
    companyfacts_refresh_days=14,
    macro_refresh_days=3,
    yf_quarterly_refresh_days=7,

    # 커버리지
    yf_quarterly_max_tickers_per_run=1000,
    fsds_quarters_each_run=24,

    # 포트폴리오
    min_dynamic_port_names=12,
    top_n=30,
    starting_capital_usd=100000.0,

    # 백테스트
    start_date="2016-01-01",
    default_backtest_years=8,

    # 역발상 공포/탐욕 (기본값 0.08 사용)
    fear_greed_live_overlay_weight=0.08,

    # 비교 백테스트 (시간 오래 걸림 — 빠르게 하려면 False)
    run_comparison_backtests=True,
)

print(f"[설정 확인]")
print(f"  기간: {cfg.start_date} ~ {cfg.end_date}")
print(f"  데이터 경로: {cfg.base_dir}")
print(f"  FRED API Key: {'설정됨' if cfg.fred_api_key else '없음'}")
print(f"  yfinance 분기 재무: {cfg.yf_quarterly_max_tickers_per_run}종목")
print(f"  FSDS 분기수: {cfg.fsds_quarters_each_run}")
print(f"  최소 포트 종목수: {cfg.min_dynamic_port_names}")
print(f"  Fear/Greed 오버레이: {cfg.fear_greed_live_overlay_weight}")
print(f"  비교 백테스트: {cfg.run_comparison_backtests}")
print("\n[OK] Cell 3 완료 — 다음 Cell 4 실행 (메인 파이프라인)")

In [ ]:
# ===== Cell 4: 파이프라인 실행 =====
# 전체 실행 시간: 약 30분~1시간 (데이터 다운로드 + 학습 + 백테스트)
import json
from dataclasses import asdict

print("=" * 60)
print("파이프라인 시작")
print("=" * 60)

result = run_default_pipeline(asdict(cfg))

print("\n" + "=" * 60)
print("파이프라인 완료")
print("=" * 60)

# 핵심 결과 출력
checks = result.get("acceptance_checks", {})
metrics = result.get("backtest_metrics", {})

print(f"\n[백테스트 결과]")
print(f"  기간: {metrics.get('months', 0)}개월")
print(f"  전략 CAGR: {metrics.get('cagr', 0):.2%}")
print(f"  벤치마크 CAGR: {metrics.get('benchmark_cagr', 0):.2%}")
print(f"  초과 수익: {metrics.get('excess_cagr', 0):.2%}")
print(f"  샤프: {metrics.get('sharpe', 0):.3f}")
print(f"  최대 낙폭: {metrics.get('max_dd', 0):.2%}")
print(f"  평균 종목수: {metrics.get('avg_stock_names', 0):.1f}")
print(f"  평균 현금: {metrics.get('avg_cash_weight', 0):.2%}")

print(f"\n[데이터 품질]")
print(f"  PIT 위반: {checks.get('pit_violation_count', '?')}")
print(f"  데이터 누수: {checks.get('leakage_ok', '?')}")
print(f"  백테스트 사용가능: {checks.get('backtest_usable', '?')}")
print(f"  펀더멘탈 커버리지: {checks.get('fundamental_coverage_ok', '?')}")

print(f"\n[포트폴리오]")
print(f"  추천 종목수: {result.get('latest_portfolio_rows', 0)}")

In [ ]:
# ===== Cell 5 (선택): 결과 파일 다운로드 =====
from google.colab import files
from pathlib import Path

out_dir = Path(cfg.base_dir) / "outputs"
report_dir = out_dir / "reports"

# (폴더, 파일명) 쌍 — outputs/ 와 outputs/reports/ 에 나뉘어 있음
download_files = [
    # outputs/ 에 있는 파일
    (out_dir, "portfolio_latest.csv"),
    (out_dir, "top30_latest.csv"),
    (out_dir, "top20_latest.csv"),
    (out_dir, "scored_latest.csv"),
    (out_dir, "weights_latest.json"),
    (out_dir, "equity_curve.csv"),
    (out_dir, "full_fundamental_rank_latest.csv"),
    # outputs/reports/ 에 있는 파일
    (report_dir, "benchmark_comparison_latest.json"),
    (report_dir, "acceptance_checks.json"),
    (report_dir, "market_adaptation_latest.json"),
    (report_dir, "portfolio_size_comparison.csv"),
    (report_dir, "ranking_quality.json"),
]

print("다운로드 가능한 파일:")
for folder, fname in download_files:
    fpath = folder / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"  [OK] {fname} ({size_kb:.1f} KB)")
        files.download(str(fpath))
    else:
        print(f"  [--] {fname} (없음)")

In [ ]:
# ===== Cell 6 (선택): 포트폴리오 상세 보기 =====
import pandas as pd
from pathlib import Path

out_dir = Path(cfg.base_dir) / "outputs"
portfolio_path = out_dir / "portfolio_latest.csv"
if portfolio_path.exists():
    df = pd.read_csv(portfolio_path)
    cols = ["rank", "ticker", "Name", "sector", "weight", "score"]
    cols = [c for c in cols if c in df.columns]
    display(df[cols].head(30))
else:
    print("portfolio_latest.csv 없음")